# Hito 2 - Notebook 05: Preparacion de Datos - Frente Salud
## Fase 3 de CRISP-DM (Data Preparation & Feature Engineering)

Orden logico: **Carga -> Limpieza -> Transformacion -> Reduccion**. La integracion de datos se realiza en el notebook 07.

In [1]:
import sys
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')
ROOT = Path.cwd()
while not (ROOT / 'src' / 'aldimi_common.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import aldimi_common as ac
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
%matplotlib inline
sns.set_theme(style='whitegrid'); plt.rcParams['figure.figsize'] = (9, 5)
pd.set_option('display.max_columns', None)
print('Raiz del proyecto:', ROOT)

Raiz del proyecto: D:\2026-01\MachLearning\finTF


## 1.4.1 Carga de Datos

In [2]:
salud = pd.read_csv(ac.DATA_RAW / ac.HEALTH_RAW_FILE)
print('Datos crudos:', salud.shape)
salud.head(3)

Datos crudos: (143194, 22)


,Patient_ID,Age,Gender,Country,WBC_Count,RBC_Count,Platelet_Count,Hemoglobin_Level,Bone_Marrow_Blasts,Genetic_Mutation,Family_History,Smoking_Status,Alcohol_Consumption,Radiation_Exposure,Infection_History,BMI,Chronic_Illness,Immune_Disorders,Ethnicity,Socioeconomic_Status,Urban_Rural,Leukemia_Status
0,1,52,Male,China,2698,5.36,262493,12.2,72,Yes,No,Yes,No,No,No,24.0,No,No,Ethnic_Group_B,Low,Rural,Negative
1,2,15,Female,China,4857,4.81,277877,11.9,97,Yes,No,No,No,No,No,28.7,No,No,Ethnic_Group_A,Low,Urban,Positive
2,3,72,Male,France,9614,5.17,319600,13.4,94,No,Yes,No,Yes,No,No,27.7,No,No,Ethnic_Group_B,Low,Urban,Negative


## 1.4.2 Limpieza de Datos (Data Cleaning)

In [3]:
print('Nulos totales:', int(salud.isna().sum().sum()))
print('Duplicados (fila completa):', int(salud.duplicated().sum()))
# Los identificadores deben ser unicos
dups_id = salud['Patient_ID'].duplicated().sum()
print('Patient_ID duplicados:', int(dups_id))
salud = salud.drop_duplicates(subset='Patient_ID').reset_index(drop=True)
print('Tras limpieza:', salud.shape)

Nulos totales: 0


Duplicados (fila completa): 0
Patient_ID duplicados: 0
Tras limpieza: (143194, 22)


### Restriccion de cohorte: poblacion pediatrico-juvenil (Age < 25)

ALDIMI es un **albergue oncologico pediatrico**; el dataset publico de Kaggle incluye pacientes adultos que no representan la poblacion atendida. Se conservan solo registros con `Age < 25` para alinear el modelo con el contexto de negocio (ninos y adolescentes con cancer).

In [4]:
antes = len(salud)
salud = ac.filter_pediatric_cohort(salud)
print(f'Cohorte pediatrico-juvenil (Age < {ac.HEALTH_MAX_AGE}): {salud.shape[0]:,} pacientes '
      f'(excluidos {antes - len(salud):,} adultos)')
print('Rango de edad:', int(salud['Age'].min()), '-', int(salud['Age'].max()))

Cohorte pediatrico-juvenil (Age < 25): 38,817 pacientes (excluidos 104,377 adultos)
Rango de edad: 1 - 24


> **Conclusion limpieza:** el dataset esta completo (sin nulos) y sin duplicados de paciente. No se requiere imputacion. Se mantiene la integridad del identificador.

## 1.4.3 Transformacion de Datos (Data Transformation)

### Feature Engineering (creacion de nuevas variables)

In [5]:
salud = ac.add_health_features(salud)
salud = ac.derive_priority_label(salud)
nuevas = ['Severidad_Clinica', 'Habitos_Riesgo', 'Riesgo_Antecedentes',
          'Vulnerabilidad_Social', 'Leucemia_Positiva', 'Edad_Rango',
          'Indice_Riesgo_Clinico', 'Score_Triage',
          'Prioridad_Score', 'Prioridad_Atencion']
salud[nuevas].head()

,Severidad_Clinica,Habitos_Riesgo,Riesgo_Antecedentes,Vulnerabilidad_Social,Leucemia_Positiva,Edad_Rango,Indice_Riesgo_Clinico,Score_Triage,Prioridad_Score,Prioridad_Atencion
0,3,0,1,1,1,Adolescente,13.5,13.361523,13.646264,Alto
1,1,1,0,2,0,Adulto_Joven,6.5,6.448510,6.000808,Bajo
2,1,1,0,2,0,Adulto_Joven,6.5,6.680310,6.860217,Bajo
3,1,1,0,0,0,Nino,3.5,3.527156,3.951471,Bajo
4,2,1,1,2,0,Adulto_Joven,10.5,10.628832,9.563503,Medio


> Se crean indices interpretables: **Severidad_Clinica**, **Habitos_Riesgo**, **Riesgo_Antecedentes**, **Vulnerabilidad_Social** (ODS 10), **Indice_Riesgo_Clinico** (composicion ponderada validada en EDA), **Score_Triage** (evaluacion inicial en admision, protocolo ALDIMI) y el objetivo **Prioridad_Atencion**.

### Manejo del desbalanceo de clases

In [6]:
dist = salud['Prioridad_Atencion'].value_counts().reindex(ac.PRIORITY_ORDER)
print(dist)
print('\nProporciones:'); print((dist / dist.sum()).round(3))

Prioridad_Atencion
Bajo     21349
Medio    11645
Alto      5823
Name: count, dtype: int64

Proporciones:
Prioridad_Atencion
Bajo     0.55
Medio    0.30
Alto     0.15
Name: count, dtype: float64


> El objetivo presenta **desbalance** (la clase 'Alto' es minoritaria). **Decision:** NO se aplica SMOTE sobre todo el dataset aqui. Se aplicara **dentro del Pipeline de validacion cruzada** en el modelado avanzado (notebook 09), solo sobre los pliegues de entrenamiento.

### Codificacion de variables categoricas (Encoding) - demostracion

In [7]:
cat_cols = salud.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c != ac.PRIORITY_TARGET]
print('Categoricas a codificar:', cat_cols)
demo_ohe = pd.get_dummies(salud[cat_cols].head(), drop_first=True)
print('Ejemplo One-Hot (shape):', demo_ohe.shape)
demo_ohe.head()

Categoricas a codificar: ['Gender', 'Country', 'Genetic_Mutation', 'Family_History', 'Smoking_Status', 'Alcohol_Consumption', 'Radiation_Exposure', 'Infection_History', 'Chronic_Illness', 'Immune_Disorders', 'Ethnicity', 'Socioeconomic_Status', 'Urban_Rural', 'Leukemia_Status', 'Edad_Rango']
Ejemplo One-Hot (shape): (5, 16)


,Gender_Male,Country_China,Country_Russia,Genetic_Mutation_Yes,Smoking_Status_Yes,Alcohol_Consumption_Yes,Radiation_Exposure_Yes,Infection_History_Yes,Chronic_Illness_Yes,Ethnicity_Ethnic_Group_B,Ethnicity_Ethnic_Group_C,Socioeconomic_Status_Medium,Urban_Rural_Urban,Leukemia_Status_Positive,Edad_Rango_Adulto_Joven,Edad_Rango_Nino
0,False,True,False,True,False,False,False,False,False,False,False,False,True,True,False,False
1,True,False,False,False,True,False,False,False,False,True,False,False,False,False,True,False
2,True,False,True,False,False,False,False,True,False,False,True,False,False,False,True,False
3,True,True,False,False,False,False,True,False,False,False,False,True,True,False,False,True
4,True,False,False,False,False,True,False,False,True,False,False,False,False,False,True,False


### Escalado de variables numericas (Scaling) - demostracion

In [8]:
from sklearn.preprocessing import StandardScaler
num_cols = salud.select_dtypes(include=np.number).columns.tolist()
num_cols = [c for c in num_cols if c not in ac.HEALTH_EXCLUDE_COLS]
demo_scaled = pd.DataFrame(StandardScaler().fit_transform(salud[num_cols]), columns=num_cols)
demo_scaled.describe().T[['mean', 'std']].round(3).head()

,mean,std
Age,-0.0,1.0
WBC_Count,-0.0,1.0
RBC_Count,-0.0,1.0
Platelet_Count,-0.0,1.0
Hemoglobin_Level,-0.0,1.0


> **Conclusion transformacion:** se elige **StandardScaler** (media 0, desviacion 1) frente a MinMaxScaler porque las variables tienen escalas muy distintas y valores extremos (el estandarizado es mas robusto ante outliers). El encoding y el escalado se **encapsulan en el Pipeline del modelado** (aplicados solo al entrenamiento en cada fold). El dataset guardado conserva las variables interpretables.

## 1.4.4 Reduccion de Datos (Data Reduction)

In [9]:
features = ac.health_feature_columns(salud)
print(f'Variables candidatas para el modelo: {len(features)}')
print('Se EXCLUYEN del modelo (identificadores / target / referencia interna):', ac.HEALTH_EXCLUDE_COLS)
# Se descarta la columna redundante original cuando existe su version _flag
redundantes = [c for c in ac.HEALTH_YESNO_COLS if f'{c}_flag' in salud.columns]
print('Columnas Yes/No con equivalente _flag (redundancia controlada):', redundantes)

Variables candidatas para el modelo: 41
Se EXCLUYEN del modelo (identificadores / target / referencia interna): ['Patient_ID', 'Prioridad_Score', 'Prioridad_Atencion']
Columnas Yes/No con equivalente _flag (redundancia controlada): ['Genetic_Mutation', 'Family_History', 'Smoking_Status', 'Alcohol_Consumption', 'Radiation_Exposure', 'Infection_History', 'Chronic_Illness', 'Immune_Disorders']


> **Conclusion reduccion:** se privilegia la **seleccion de variables interpretables** sobre la compresion PCA (que reduciria la interpretabilidad clinica). Se excluyen identificadores, el target y `Prioridad_Score` (valoracion integral de referencia, no disponible en inferencia).

## Guardado del dataset preparado

In [10]:
ac.DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
salida = ac.DATA_PROCESSED / ac.HEALTH_PROCESSED_FILE
salud.to_csv(salida, index=False)
print('Guardado:', salida, salud.shape)

Guardado: D:\2026-01\MachLearning\finTF\data\processed\Dataset_ALDIMI_Salud_Preparado.csv (38817, 44)


> **Salida:** `data/processed/Dataset_ALDIMI_Salud_Preparado.csv`, listo para integracion en BD (07) y modelado (08-09).